In [2]:
# %%
# ChatGPT-like UI (Dark Mode) + Streaming + Logs Panel + Ollama Agent
# 실행:
# pip install streamlit requests tavily-python
# streamlit run this_file.py

import streamlit as st
import requests
from tavily import TavilyClient
import time

# ================= Config =================
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "deepseek-coder:6.7b"
TAVILY_API_KEY = "YOUR_TAVILY_API_KEY"

client = TavilyClient(api_key=TAVILY_API_KEY)

st.set_page_config(page_title="Local ChatGPT", layout="wide")

# ================= Styles (Dark UI) =================
st.markdown("""
<style>
html, body, [class*="css"]  {
    background-color: #0f172a;
    color: #e5e7eb;
}
.stChatMessage {
    border-radius: 16px;
    padding: 12px;
}
</style>
""", unsafe_allow_html=True)

# ================= State =================
if "messages" not in st.session_state:
    st.session_state.messages = []
if "logs" not in st.session_state:
    st.session_state.logs = []

# ================= LLM (Streaming) =================
def llm_stream(prompt):
    res = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "prompt": prompt,
        "stream": True
    }, stream=True)

    for line in res.iter_lines():
        if line:
            try:
                data = line.decode("utf-8")
                if '"response"' in data:
                    chunk = eval(data).get("response", "")
                    yield chunk
            except:
                continue

# ================= Search =================
def search(query):
    result = client.search(query=query, max_results=3)
    docs = []
    for r in result['results']:
        docs.append(f"[{r['url']}]\n{r['content']}")
    st.session_state.logs.append({"type": "search", "data": docs})
    return "\n\n".join(docs)

# ================= Agent =================
def run_agent(query):
    context = search(query)

    prompt = f"""
다음 정보 기반으로 답변:
{context}

질문: {query}

조건:
- 핵심만
- 정확하게
"""

    return prompt

# ================= Layout =================
col1, col2 = st.columns([3, 1])

# ===== Left: Chat =====
with col1:
    st.title("💬 Local ChatGPT")

    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    if prompt := st.chat_input("질문 입력"):
        st.session_state.messages.append({"role": "user", "content": prompt})

        with st.chat_message("user"):
            st.markdown(prompt)

        with st.chat_message("assistant"):
            placeholder = st.empty()
            full_text = ""

            agent_prompt = run_agent(prompt)

            for chunk in llm_stream(agent_prompt):
                full_text += chunk
                placeholder.markdown(full_text)

            st.session_state.messages.append({"role": "assistant", "content": full_text})

# ===== Right: Logs Panel =====
with col2:
    st.title("📊 Logs")

    for log in reversed(st.session_state.logs):
        if log["type"] == "search":
            st.markdown("### 🔍 Search Results")
            for d in log["data"]:
                st.markdown(d[:300] + "...")

# ===== Sidebar =====
st.sidebar.title("⚙️ Settings")

new_model = st.sidebar.text_input("Model", MODEL)
if new_model:
    MODEL = new_model

if st.sidebar.button("Clear Chat"):
    st.session_state.messages = []

if st.sidebar.button("Clear Logs"):
    st.session_state.logs = []


2026-05-01 17:28:22.072 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-01 17:28:22.074 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-01 17:28:22.113 
  command:

    streamlit run /Users/wondongsoo/miniconda3/conda_envs/AiMathematics/lib/python3.10/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-05-01 17:28:22.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-01 17:28:22.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-01 17:28:22.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-01 17:28:22.114 Session state does not function when running